In [ ]:
import os
import pandas as pd
import cobra
from cobra.io import read_sbml_model
from sklearn import metrics

def check_uptake(model, ex_id, tol=1e-4):
    for rxn in model.reactions:
        if rxn.id.startswith('EX_'):
            rxn.lower_bound = -1000  # uptake
            rxn.upper_bound = 1000   # secretion
    if ex_id not in model.reactions:
        return False, 0
    model.objective = model.reactions.get_by_id(ex_id)
    model.objective_direction = 'min'
    solution = model.optimize()
    flux = solution.objective_value if solution.status == 'optimal' else 0
    can_uptake = flux < -tol
    return can_uptake, flux


agora2_mapdict = {
    'maltose':'EX_malt(e)', 'sucrose':'EX_sucr(e)','nitrate':'EX_no3(e)',
    'urea':'EX_urea(e)','esculin':'EX_escul(e)', 'arginine':'EX_arg_L(e)', 
    'raffinose':'EX_raffin(e)', 'lactose':'EX_lcts(e)'
}


input_dir = '../data/benchmark_data/agora/'
output_dir = 'agora_result'
os.makedirs(output_dir, exist_ok=True)


for compound, exid in agora2_mapdict.items():
    test_data = pd.read_csv(f'{input_dir}{compound}.csv')
    c_result = test_data[['Species', compound]].copy()
    pre = []
    objective_values = []
    for p in test_data['AGORA2 model path']:
        try:
            model = read_sbml_model(p)
            can_uptake, flux = check_uptake(model, exid)
            pre.append(1 if can_uptake else 0)
            objective_values.append(flux)
        except Exception as e:
            print(f"Error processing {p}: {e}")
            pre.append('NA')
            objective_values.append('NA')

    c_result['pre'] = pre
    c_result['objective_value'] = objective_values
    c_result.to_csv(f'{output_dir}{compound}.csv', index=False)
    c_result['True_Label'] = c_result[compound].apply(lambda x: x if x in [0, 1] else 'NA')
    c_result = c_result[c_result['pre'] != 'NA']
    c_result['True_Label'] = c_result['True_Label'].astype(int)
    c_result['pre'] = c_result['pre'].astype(int)
    
    metabolism_labels = c_result['True_Label'].tolist()
    predictions_ls = c_result['pre'].tolist()
    
    summary = {
        'Compound': compound,
        'Tested_Models': len(c_result)
    }
        
    summary['Accuracy'] = metrics.accuracy_score(metabolism_labels, predictions_ls)
    summary['Precision'] = metrics.precision_score(metabolism_labels, predictions_ls, zero_division=0)
    summary['Recall'] = metrics.recall_score(metabolism_labels, predictions_ls, zero_division=0)
    summary['F1_Score'] = metrics.f1_score(metabolism_labels, predictions_ls, zero_division=0)
    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(f'{output_dir}{compound}_summary.csv', index=False)

In [ ]:
def get_compound_ex_id(compound_name, carbon_source):
        exid = carbon_source[carbon_source["name"] == compound_name]["exid"].values[0]
        exid = exid.replace("(e)", "_e")
        return exid

input_dir = '../data/benchmark_data/auto_gem'
output_dir = 'auto_gem_result'
os.makedirs(output_dir, exist_ok=True)
compound2id = pd.read_csv("../data/Compound2ID.csv")

for compound in compound2id['compound']:
    test_data = pd.read_csv(f'{input_dir}{compound}.csv')
    c_result = test_data[['Species', compound]].copy()
    exid = get_compound_ex_id(compound, compound2id)
    pre = []
    objective_values = []
    for p in test_data['carve_data_path']:
        try:
            model = read_sbml_model(p)
            can_uptake, flux = check_uptake(model, exid)
            pre.append(1 if can_uptake else 0)
            objective_values.append(flux)
        except Exception as e:
            print(f"Error processing {p}: {e}")
            pre.append('NA')
            objective_values.append('NA')

    c_result['pre'] = pre
    c_result['objective_value'] = objective_values
    c_result.to_csv(f'{output_dir}{compound}.csv', index=False)
    c_result['True_Label'] = c_result[compound].apply(lambda x: x if x in [0, 1] else 'NA')
    c_result = c_result[c_result['pre'] != 'NA'].copy()
    c_result['True_Label'] = c_result['True_Label'].astype(int)
    c_result['pre'] = c_result['pre'].astype(int)
    
    metabolism_labels = c_result['True_Label'].tolist()
    predictions_ls = c_result['pre'].tolist()
    
    summary = {
        'Compound': compound,
        'Tested_Models': len(c_result)
    }
    
    summary['Accuracy'] = metrics.accuracy_score(metabolism_labels, predictions_ls)
    summary['Precision'] = metrics.precision_score(metabolism_labels, predictions_ls, zero_division=0)
    summary['Recall'] = metrics.recall_score(metabolism_labels, predictions_ls, zero_division=0)
    summary['F1_Score'] = metrics.f1_score(metabolism_labels, predictions_ls, zero_division=0)
    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(f'{output_dir}{compound}_summary.csv', index=False)

In [ ]:
def get_gapseq_compound_ex_id(compound_name, carbon_source):
    exid = carbon_source[carbon_source["name"] == compound_name]["exid_seed"].values[0]
    return exid

input_dir = '../data/benchmark_data/auto_gem'
output_dir = 'auto_gem_result'
os.makedirs(output_dir, exist_ok=True)

for compound in compound2id['compound']:
    test_data = pd.read_csv(f'{input_dir}{compound}.csv')
    c_result = test_data[['Species', compound]].copy()
    exid = get_gapseq_compound_ex_id(compound, compound2id)
    pre = []
    objective_values = []
    for p in test_data['gapseq_path']:
        try:
            model = read_sbml_model(p)
            can_uptake, flux = check_uptake(model, exid)
            pre.append(1 if can_uptake else 0)
            objective_values.append(flux)
        except Exception as e:
            print(f"Error processing {p}: {e}")
            pre.append('NA')
            objective_values.append('NA')

    c_result['pre'] = pre
    c_result['objective_value'] = objective_values
    c_result.to_csv(f'{output_dir}{compound}.csv', index=False)
    c_result['True_Label'] = c_result[compound].apply(lambda x: x if x in [0, 1] else 'NA')
    c_result = c_result[c_result['pre'] != 'NA'].copy()
    c_result['True_Label'] = c_result['True_Label'].astype(int)
    c_result['pre'] = c_result['pre'].astype(int)
    metabolism_labels = c_result['True_Label'].tolist()
    predictions_ls = c_result['pre'].tolist()

    summary = {
        'Compound': compound,
        'Tested_Models': len(c_result)
    }
    
    summary['Accuracy'] = metrics.accuracy_score(metabolism_labels, predictions_ls)
    summary['Precision'] = metrics.precision_score(metabolism_labels, predictions_ls, zero_division=0)
    summary['Recall'] = metrics.recall_score(metabolism_labels, predictions_ls, zero_division=0)
    summary['F1_Score'] = metrics.f1_score(metabolism_labels, predictions_ls, zero_division=0)
    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(f'{output_dir}{compound}_summary.csv', index=False)